## Context

This notebook accompanies User Guide section 15 (Constrained LLM).  It exercises the three-channel governance contract, interpret, propose_next, extract_claims, twice: first with `MockConstrainedLLM` (deterministic, no API key) and then with the real `ConstrainedLLM` wired to `AnthropicBackend`.  The two classes share a contract: same method signatures, same `Proposal` and `GatedClaim` shapes, same fail-closed default when the GMS store has no trained model.  The real class adds a non-negotiable: when `ANTHROPIC_API_KEY` is missing the constructor raises `AnthropicAPIKeyError` rather than falling back to canned text.

In [ ]:
# Phase 1B preamble: bootstrap runtime + load shared helpers
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# tutorials/user_guide/notebooks/_shared.py. Works in both the source
# clone (knowlytix/GMS) and the public mirror clone (knowlytix/gmsh).
import os
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'tutorials' / 'user_guide' / 'notebooks' / '_shared.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from tutorials.user_guide.notebooks import _helpers_phase1 as h1
h1.bootstrap_runtime()  # stubs modeva if absent
from tutorials.user_guide.notebooks import _shared as ug
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')


### Concept: LLM proposes, GMS approves

The pattern routes every LLM output through one of three governance channels.  Channel 1, **interpretation**, is ungated and produces display-only narrative.  Channel 2, **proposals**, is plausibility-gated through `LogicVerifier.verify_action`; each proposed tool call is scored against the geodesic threshold and rejected when implausible.  Channel 3, **claims**, is contradiction-gated through `LogicVerifier.verify_domain_triples`; factual claims extracted from the narrative are checked against tension with the existing manifold and rejected when they would introduce a contradiction.

Both `MockConstrainedLLM` and `ConstrainedLLM` enforce fail-closed semantics: when `store.model` is `None` every proposal and every claim is rejected with reason `fail-closed: no trained GMS model`.  This prevents the authority leak where an LLM operates without geometric verification.  The classes are interchangeable wherever downstream code consumes the three-channel surface.

Section A below walks through the mock against the prebuilt `ds_practices_store` fixture.  Section B then instantiates the real `ConstrainedLLM` with `AnthropicBackend` if `ANTHROPIC_API_KEY` is set, or demonstrates the loud failure path otherwise.  No mock fallback: a missing key is a fatal configuration error.

## Section A — `MockConstrainedLLM`

Build a verifier and a mock constrained LLM against the shared fixture.  The verifier is configured with `min_plausibility=1.5` (the default published throughout the user guide).  The mock then exercises all three channels.

In [ ]:
from knowlytix.knowledge.config import VerifyConfig
from knowlytix.harness._verifier import LogicVerifier
from knowlytix.harness.governance.governed_llm import (
    MockConstrainedLLM,
    Proposal,
    GatedClaim,
    format_proposal_report,
    format_claim_report,
)

store = ug.load_ds_practices_store()
verifier = LogicVerifier(store,
                         VerifyConfig(min_plausibility=1.5))
mock_llm = MockConstrainedLLM(store, verifier)
print('store.model is not None ->',
      store.model is not None)
print('mock_llm ready: type =', type(mock_llm).__name__)


In [ ]:
# Channel 1: ungated interpretation (display-only)
raw_result = {
    'accuracy': 0.92, 'f1': 0.91,
    'model_name': 'RandomForest', 'n_epochs': 100,
}
narrative = mock_llm.interpret(
    'train_model', raw_result, context='training_phase')
print('Narrative:')
print(' ', narrative)


In [ ]:
# Channel 2: plausibility-gated proposals
mock_proposals = mock_llm.propose_next(
    context='training',
    completed=['load_dataset', 'preprocess_data'],
)
print(format_proposal_report(mock_proposals))
mock_proposals_total = len(mock_proposals)
mock_approved = sum(1 for p in mock_proposals if p.approved)
print(f'\nMock total={mock_proposals_total} '
      f'approved={mock_approved}')


In [ ]:
# Channel 3: contradiction-gated claims
mock_claims = mock_llm.extract_claims(narrative)
print(format_claim_report(mock_claims))
mock_claims_total = len(mock_claims)
mock_claims_approved = sum(1 for c in mock_claims if c.approved)
print(f'\nMock claims={mock_claims_total} '
      f'approved={mock_claims_approved}')


In [ ]:
# Visualization 1: approved-vs-rejected for proposals
import matplotlib.pyplot as plt

tools = [p.tool for p in mock_proposals]
approved = [1 if p.approved else 0 for p in mock_proposals]
rejected = [0 if p.approved else 1 for p in mock_proposals]

fig, ax = plt.subplots(figsize=(7.5, 3.6))
x = list(range(len(tools)))
ax.bar(x, approved, color='tab:green', label='approved')
ax.bar(x, rejected, bottom=approved,
       color='tab:red', label='rejected')
ax.set_xticks(x)
ax.set_xticklabels(tools, rotation=20, ha='right')
ax.set_ylabel('count')
ax.set_ylim(0, 1.4)
ax.set_title('MockConstrainedLLM: per-proposal verdict')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()


## Section B — Real `ConstrainedLLM` (Anthropic backend)

Same contract, real LLM.  Every output flows through the same three channels and produces the same `Proposal` / `GatedClaim` shapes.  When `ANTHROPIC_API_KEY` is missing the constructor raises `AnthropicAPIKeyError` and we surface that exception rather than falling back to a mock.  The cell below branches on whether a key is resolvable; either branch is a successful demonstration of the documented behavior.

In [ ]:
from knowlytix.harness.governance.api_key import (
    AnthropicAPIKeyError,
    get_anthropic_api_key,
)
from knowlytix.knowledge.llm_backend import AnthropicBackend
from knowlytix.harness.governance.governed_llm import ConstrainedLLM

real_llm = None
real_proposals = None
real_claims = None
real_narrative = None
key_available = False
try:
    _ = get_anthropic_api_key()
    key_available = True
except AnthropicAPIKeyError as exc:
    print('No API key resolvable; demonstrating loud '
          'failure path.')
    print(f'  exception type: {type(exc).__name__}')
    print(f'  message head:   {str(exc)[:80]}...')

print(f'key_available = {key_available}')


In [ ]:
# Branch B1: key set -> real ConstrainedLLM, three channels
if key_available:
    backend = AnthropicBackend(model='claude-opus-4-7')
    real_llm = ConstrainedLLM(
        store=store,
        verifier=verifier,
        llm=backend,
        allowed_tools=[
            'load_dataset', 'preprocess_data',
            'train_model', 'evaluate_model',
            'diagnose_accuracy', 'explain_model',
        ],
    )
    print('real_llm ready: type =',
          type(real_llm).__name__)
    real_narrative = real_llm.interpret(
        'train_model', raw_result,
        context='training_phase')
    print('Real narrative head:')
    print(' ', (real_narrative or '')[:240])
    real_proposals = real_llm.propose_next(
        context='training',
        completed=['load_dataset', 'preprocess_data'],
    )
    print()
    print(format_proposal_report(real_proposals))
    real_claims = real_llm.extract_claims(
        real_narrative or narrative)
    print()
    print(format_claim_report(real_claims))
else:
    print('skipped real LLM channels: '
          'ANTHROPIC_API_KEY not resolvable.')


In [ ]:
# Branch B2: explicit demonstration of loud-failure path.
# Even when a key IS configured globally, we can show the
# documented exception by transiently masking it.  This is
# the path tests run to assert no mock-LLM fallback exists.
_saved_env = os.environ.pop('ANTHROPIC_API_KEY', None)
raised = None
try:
    # Temporarily shadow the password-store fallback as well
    # by routing through get_anthropic_api_key directly.
    from knowlytix.harness.governance import api_key as _ak
    _orig = _ak.get_anthropic_api_key
    def _no_key():
        raise AnthropicAPIKeyError(
            'simulated missing key for tutorial demo')
    _ak.get_anthropic_api_key = _no_key
    try:
        _bad = AnthropicBackend(model='claude-opus-4-7')
    except AnthropicAPIKeyError as exc:
        raised = exc
    finally:
        _ak.get_anthropic_api_key = _orig
finally:
    if _saved_env is not None:
        os.environ['ANTHROPIC_API_KEY'] = _saved_env

print('Expected exception:',
      type(raised).__name__ if raised else 'none')
if raised is not None:
    print(' ', str(raised)[:120])


In [ ]:
# Visualization 2: side-by-side mock vs real proposal pass rate.
# When the real LLM did not run (no key) the right bar is empty.
import matplotlib.pyplot as plt

labels = ['mock', 'real']
approved_counts = [
    sum(1 for p in mock_proposals if p.approved),
    sum(1 for p in (real_proposals or []) if p.approved),
]
rejected_counts = [
    sum(1 for p in mock_proposals if not p.approved),
    sum(1 for p in (real_proposals or []) if not p.approved),
]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
x = [0, 1]
ax.bar(x, approved_counts,
       color='tab:green', label='approved')
ax.bar(x, rejected_counts, bottom=approved_counts,
       color='tab:red', label='rejected')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('proposal count')
ax.set_title('Mock vs real ConstrainedLLM proposal verdict')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()


### Exercises

1. Run `mock_llm.propose_next` from a context the store   has not seen, e.g. `context='quantum_chemistry'`.    How does `LogicVerifier` respond when the relation   triple is unknown?  Look at `Proposal.score`.
2. Construct a `GatedClaim` candidate with a   contradicting numeric value (for instance,   `RandomForest has_accuracy 0.0` against a fixture   that already records accuracy 1.0).  Pass it through   `verifier.verify_domain_triples`.  Does tension cross   the contradiction band $\sqrt{2}$?
3. With `ANTHROPIC_API_KEY` set, compare   `format_proposal_report(mock_proposals)` against   `format_proposal_report(real_proposals)`.  Identify   one tool the real backend proposes that the mock did   not, or vice versa.  What does the geometric gate say   about the new proposal?

### What you should now be able to do

- Construct a `MockConstrainedLLM` against a built   store and run all three channels.
- Read `Proposal.approved`, `Proposal.score`, and   `Proposal.rejection_reason` to interpret a per-tool   verdict.
- Construct a real `ConstrainedLLM` wired to   `AnthropicBackend` and recognise that the two   classes are interchangeable behind the same   three-channel contract.
- Demonstrate the loud-failure path   (`AnthropicAPIKeyError`) and explain why no mock   fallback exists in this codepath.

### Cross-references

- USER_GUIDE.md section 15 (Constrained LLM)
- `knowlytix/harness/governance/governed_llm.py:MockConstrainedLLM`   and `knowlytix/harness/governance/governed_llm.py:ConstrainedLLM`
- `knowlytix/knowledge/llm_backend.py:AnthropicBackend`
- `knowlytix/harness/governance/api_key.py:AnthropicAPIKeyError`
- `gms_agent_builder_ext/chapters/ch11.tex` for the   constrained-LLM chapter in book form

In [ ]:
# Final assertions.  Mock side is verified; real side
# is verified only when the API key is resolvable.
# TODO(p7-execute): tighten if hybrid run.
assert mock_proposals_total >= 4, (
    f'mock returned {mock_proposals_total} proposals; '
    f'expected >= 4 (3 next-step + at least 1 dubious '
    f'+ extras)'
)
assert all(
    isinstance(p, Proposal) for p in mock_proposals
), 'mock_proposals must be a list of Proposal'
assert all(
    isinstance(c, GatedClaim) for c in mock_claims
), 'mock_claims must be a list of GatedClaim'
if key_available:
    assert real_llm is not None, (
        'real_llm constructor returned None despite '
        'an available API key'
    )
    assert hasattr(real_llm, 'interpret'), (
        'real_llm missing interpret channel'
    )
    assert hasattr(real_llm, 'propose_next'), (
        'real_llm missing propose_next channel'
    )
    assert hasattr(real_llm, 'extract_claims'), (
        'real_llm missing extract_claims channel'
    )
print(f'OK: mock_proposals={mock_proposals_total}, '
      f'mock_claims={mock_claims_total}, '
      f'key_available={key_available}.')
